# Data Merging: Tesla Stock Data, Web Scraping News and REST API News

In this notebook, the cleaned Tesla stock data is merged with two news datasets:

1. Tesla news from web scraping
2. Tesla news from the Alpha Vantage REST API

The goal is to create one combined dataset that contains stock price movements, news counts and sentiment information per date.

# 2. Imports

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt

# 3. Load Data

In [ ]:
stock_df = pd.read_csv("../data/processed/tesla_stock_data_cleaned.csv")
scraped_news_df = pd.read_csv("../data/processed/tesla_news_cleaned.csv")
api_news_df = pd.read_csv("../data/processed/alpha_vantage_tesla_news_cleaned.csv")

print("Stock data:", stock_df.shape)
print("Scraped news data:", scraped_news_df.shape)
print("API news data:", api_news_df.shape)

display(stock_df.head())
display(scraped_news_df.head())
display(api_news_df.head())

# 4. Convert Date Columns

In [ ]:
stock_df["Date"] = pd.to_datetime(stock_df["Date"])
scraped_news_df["date"] = pd.to_datetime(scraped_news_df["date"])
api_news_df["date"] = pd.to_datetime(api_news_df["date"])

print("Stock date range:", stock_df["Date"].min(), "to", stock_df["Date"].max())
print("Scraped news date range:", scraped_news_df["date"].min(), "to", scraped_news_df["date"].max())
print("API news date range:", api_news_df["date"].min(), "to", api_news_df["date"].max())

# 5. Aggregate Web Scraping News per Day

In [ ]:
scraped_daily = scraped_news_df.groupby("date").agg(
    scraped_news_count=("title", "count"),
    scraped_sources=("source", lambda x: ", ".join(sorted(set(x.dropna().astype(str))))),
    scraped_titles=("title", lambda x: " | ".join(x.dropna().astype(str).head(3)))
).reset_index()

display(scraped_daily.head())

# 6. Aggregate REST API News per Day

In [ ]:
api_daily = api_news_df.groupby("date").agg(
    api_news_count=("title", "count"),
    avg_api_sentiment=("overall_sentiment_score", "mean"),
    api_sentiment_labels=("overall_sentiment_label", lambda x: ", ".join(sorted(set(x.dropna().astype(str))))),
    api_titles=("title", lambda x: " | ".join(x.dropna().astype(str).head(3)))
).reset_index()

display(api_daily.head())

# 7. Filter Stock Data to Relevant News Period

In [ ]:
min_date = min(scraped_news_df["date"].min(), api_news_df["date"].min())
max_date = max(scraped_news_df["date"].max(), api_news_df["date"].max())

stock_filtered = stock_df[
    (stock_df["Date"] >= min_date) &
    (stock_df["Date"] <= max_date)
].copy()

print("Filtered stock data:", stock_filtered.shape)
print("Used date range:", min_date, "to", max_date)

# 8. Merge All Data

In [ ]:
merged_df = stock_filtered.merge(
    scraped_daily,
    left_on="Date",
    right_on="date",
    how="left"
)

merged_df = merged_df.drop(columns=["date"])

merged_df = merged_df.merge(
    api_daily,
    left_on="Date",
    right_on="date",
    how="left"
)

merged_df = merged_df.drop(columns=["date"])

display(merged_df.head())

# 9. Fill Missing Values

In [ ]:
merged_df["scraped_news_count"] = merged_df["scraped_news_count"].fillna(0)
merged_df["api_news_count"] = merged_df["api_news_count"].fillna(0)
merged_df["avg_api_sentiment"] = merged_df["avg_api_sentiment"].fillna(0)

text_columns = [
    "scraped_sources",
    "scraped_titles",
    "api_sentiment_labels",
    "api_titles"
]

for col in text_columns:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].fillna("")

print("Final merged shape:", merged_df.shape)
display(merged_df.head())

# 10. Save Merged Dataset

In [ ]:
os.makedirs("../data/results", exist_ok=True)

merged_df.to_csv("../data/results/tesla_stock_news_merged.csv", index=False)

print("Saved merged data to ../data/results/tesla_stock_news_merged.csv")
print("Final shape:", merged_df.shape)

# 11. Quick Check

In [ ]:
check_df = pd.read_csv("../data/results/tesla_stock_news_merged.csv")

print("File loaded successfully.")
print("Rows:", len(check_df))
print("Columns:", list(check_df.columns))

display(check_df.head())

# 12. Visualization: News Count vs Daily Return

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(merged_df["scraped_news_count"], merged_df["Daily_Return"])
plt.title("Scraped News Count vs Tesla Daily Return")
plt.xlabel("Scraped News Count")
plt.ylabel("Daily Return")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(merged_df["api_news_count"], merged_df["Daily_Return"])
plt.title("API News Count vs Tesla Daily Return")
plt.xlabel("API News Count")
plt.ylabel("Daily Return")
plt.show()

# 13. Visualization: Sentiment vs Daily Return

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(merged_df["avg_api_sentiment"], merged_df["Daily_Return"])
plt.title("Average API Sentiment vs Tesla Daily Return")
plt.xlabel("Average API Sentiment")
plt.ylabel("Daily Return")
plt.show()

# 14. Correlation

In [ ]:
print("Correlation scraped news count vs daily return:")
print(merged_df["scraped_news_count"].corr(merged_df["Daily_Return"]))

print("\nCorrelation API news count vs daily return:")
print(merged_df["api_news_count"].corr(merged_df["Daily_Return"]))

print("\nCorrelation API sentiment vs daily return:")
print(merged_df["avg_api_sentiment"].corr(merged_df["Daily_Return"]))

# 15. Short Conclusion

In [ ]:
print("The merged dataset combines Tesla stock prices, daily returns, scraped news counts, API news counts and API sentiment values.")
print("This dataset can now be used for further analysis with Spark and for checking whether news activity or sentiment is related to stock price movements.")